In [0]:
spark.conf.set(
    "fs.azure.account.key.nehastrgacc.dfs.core.windows.net",
    ""
)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
silver_path="abfss://silver@nehastrgacc.dfs.core.windows.net/CleanDataSet"
silver = spark.read.format("delta").load(silver_path)

gold_df = (
    silver
    .groupBy("Category")
    .agg(
        sum("Quantity")
        .alias("Total_Quantity"),

        sum("Amount")
        .alias("Total_Revenue"),

        count("OrderID")
        .alias("Total_Orders")
    )
)

display(gold_df)

gold_path="abfss://gold@nehastrgacc.dfs.core.windows.net/Kpis"


gold_df.write \
.format("delta") \
.mode("overwrite") \
.partitionBy("Category") \
.save(gold_path)

In [0]:
new_sales = [
(1011,"2025-01-06","C011","Daniel",
 "Laptop","Electronics",
 1,50000,50000,"East")
]


columns=[
"OrderID",
"order_date",
"CustomerID",
"customer_name",
"Product",
"Category",
"Quantity",
"UnitPrice",
"Amount",
"Region"
]


updates = spark.createDataFrame(
new_sales,
columns
)

# Convert datatypes to match Silver table
updates = (
    updates
    .withColumn("OrderID", col("OrderID").cast("int"))
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("UnitPrice", col("UnitPrice").cast("int"))
    .withColumn("Amount", col("Amount").cast("int"))
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
)

In [0]:
from delta.tables import *

delta_table = DeltaTable.forPath(spark,silver_path)

(delta_table.alias("old")
.merge(
updates.alias("new"),
"old.OrderID = new.OrderID")
.whenMatchedUpdateAll()
.whenNotMatchedInsertAll()
.execute()
)

In [0]:
history = (
spark.sql(
f"""
DESCRIBE HISTORY delta.`{silver_path}`
"""
)
)

display(history)

In [0]:
old_data = (spark.read
.format("delta")
.option("versionAsOf",0)
.load(silver_path)
)

display(old_data)

In [0]:
df=spark.read.format("delta").load(silver_path)
df.write.format("delta").mode("overwrite").saveAsTable("sales")

In [0]:
import time

start=time.time()
df.count()
end=time.time()

print(end-start)

In [0]:
%sql
OPTIMIZE sales

In [0]:
%sql
OPTIMIZE sales
ZORDER BY (Category)

In [0]:
gold_df.cache()

gold_df.count()

In [0]:
import time

start=time.time()
df.count()
end=time.time()

print(end-start)

In [0]:
gold_df.explain(True)

In [0]:
bronze_path="abfss://bronze@nehastrgacc.dfs.core.windows.net/DataSet"
df=spark.read.format("delta").load(bronze_path)

print(df.count())

print(df.dropDuplicates().count())

print(df.filter(df.Amount.isNull()).count())